In [ ]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from ngsolve.solvers import NewtonMinimization
import ipywidgets as widgets

In [40]:
r = 0.2
l = 1

q0_deg = 0

left = MoveTo(-r, 0).Rectangle(r, l).Face()
right = MoveTo(0, 0).Rectangle(r, l).Face()
bar = left + right
bar.name = "pendulum"
#bar = MoveTo(-r,0).Rectangle(2*r, l).Face()
bar.mass
bar.edges.Min(Y).name="rotation"
bar.edges.Min(Y).maxh=r/10
bar.faces.maxh=r/5
bar = bar.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), 180)
bar = bar.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), q0_deg)
bar.name = "pendulum"
geo = bar
geo = OCCGeometry(geo, dim=2)
mesh = Mesh(geo.GenerateMesh(maxh=0.1, quad_dominated=False))
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [39]:
bar.vertices

In [33]:
# Get all nodes within the mesh.
# Different ngsolve/netgen versions expose node coordinates via different APIs,
# so try a few fallbacks to be robust.
if hasattr(mesh, "Points"):
	nodes = mesh.Points()
elif hasattr(mesh, "ngmesh") and hasattr(mesh.ngmesh, "Points"):
	nodes = mesh.ngmesh.Points()
elif hasattr(mesh, "Vertices"):
	# mesh.Vertices() may return vertex objects; try to extract coordinates
	verts = mesh.Vertices()
	try:
		nodes = [v.Point() for v in verts]
	except Exception:
		try:
			nodes = [v.point for v in verts]
		except Exception:
			nodes = verts
else:
	raise AttributeError("Could not obtain mesh nodes: Mesh has no Nodes/Points/Vertices attributes")

In [38]:
node_0 = nodes[1]
type(node_0)

netgen.libngpy._meshing.MeshPoint

In [31]:
# Get node at center of rotation (0,0)
center_node = None

for node in nodes:
    coord = np.array(node)
    if np.linalg.norm(coord) < 1e-6:
        center_node = node
        break

TypeError: unsupported operand type(s) for *: 'netgen.libngpy._meshing.MeshPoint' and 'netgen.libngpy._meshing.MeshPoint'

In [ ]:
E = 2.1e11
nu = 0.2
rho = 7800
thickness = 0.3
rhoA = rho * thickness
g=0

lam = (E*nu)/((1+nu)*(1-2*nu))
mu = E/(2*(1+nu))

def C(u):
    F = Id(u.dim) + Grad(u)
    return F.trans * F
    
def neo_hookean( C, u):
    return 0.5*mu*(Trace(C-Id(u.dim)) + 2*mu/lam*Det(C)**(-lam/2/mu)-1)


In [ ]:
V = VectorH1(mesh, order=2)
Q = NumberSpace(mesh, definedon=mesh.Boundaries("rotation"))

fes = V * Q**2

(u, q), (v, p) = fes.TnT()

In [ ]:
gf_u = GridFunction(fes)
gf_v = GridFunction(fes)
gf_a = GridFunction(fes)

gf_uold = GridFunction(fes)
gf_vold = GridFunction(fes)
gf_aold = GridFunction(fes)

In [ ]:
# --- find the pivot from the rotation edge barycenter (reference geometry)
# L_edge = Integrate(1, mesh, definedon=mesh.Boundaries("rotation"))
# cx = Integrate(x, mesh, definedon=mesh.Boundaries("rotation")) / L_edge
# cy = Integrate(y, mesh, definedon=mesh.Boundaries("rotation")) / L_edge
cx = cy = 0

# use THAT pivot for the couple definition
r = CF((x - cx, y - cy))
rperp = CF((-r[1], r[0]))
I_edge = Integrate( InnerProduct(r, r), mesh, definedon=mesh.Boundaries("rotation") )


In [ ]:
def rigid_proxy(u):
        # Calculate center of mass position (reference + displacement)
        r_cm_x_num = Integrate( rhoA * (r[0] + u[0]), mesh, definedon=mesh.Materials("pendulum") )
        r_cm_x_denom = Integrate( rhoA, mesh, definedon=mesh.Materials("pendulum") )
        r_cm_x = r_cm_x_num / r_cm_x_denom

        r_cm_y_num = Integrate( rhoA * (r[1] + u[1]), mesh, definedon=mesh.Materials("pendulum") )
        r_cm_y_denom = Integrate( rhoA, mesh, definedon=mesh.Materials("pendulum") )
        r_cm_y = r_cm_y_num / r_cm_y_denom
        
        # Calculate angle using atan2 for full range
        theta = np.arctan2(r_cm_x, -r_cm_y) # initial position at 180 deg
        
        return theta

area = Integrate(1, mesh, definedon=mesh.Materials("pendulum"))
mass = area * rho * thickness

J_area = Integrate(rhoA * (r[0]*r[0] + r[1]*r[1]),mesh, definedon=mesh.Materials("pendulum"))
inertia = J_area * thickness

M_phys = 1000 # Nm
M_per_thickness = M_phys / thickness

torque = Parameter(0.0)


In [ ]:
bfa = BilinearForm(fes)

bfa += Variation(neo_hookean(C(u), u)*dx).Compile()

bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('rotation')

tau = 0.001
vel_new = 2/tau * (u-gf_uold.components[0]) - gf_vold.components[0]
acc_new = 2/tau * (vel_new-gf_vold.components[0]) - gf_aold.components[0]

bfa += rhoA * InnerProduct(acc_new, v) * dx
bfa += InnerProduct(CF((0, rhoA * g)), v) * dx("pendulum")

# Add external moment (linear in the test function v):
bfa += torque * InnerProduct(rperp, v) * ds("rotation")


In [ ]:
t = 0
t_end = 1

scene = Draw(gf_u.components[0], mesh, "Displacement", deformation=gf_u.components[0], autoscale=True)

t_vals = []
theta_vals = []

time_widget = widgets.FloatText(value=t, description='Time:', disabled=True)
display(time_widget)

theta_widget = widgets.FloatText(value=0.0, description='Angle (deg):', disabled=True)
display(theta_widget)

In [ ]:
M_phys = 5000 * 2
with TaskManager():
    while t < t_end:
        # Record angle over time
        theta = rigid_proxy(gf_u.components[0])
        t_vals.append(t)
        theta_vals.append(theta)
        time_widget.value = t
        theta_widget.value = np.rad2deg(theta)
        
        # Time step update
        gf_uold.vec[:] = gf_u.vec
        gf_vold.vec[:] = gf_v.vec
        gf_aold.vec[:] = gf_a.vec
        t += tau
        
        if t < 0.25 * t_end:
            torque.Set( M_phys / thickness / I_edge )
        elif t < 0.75 * t_end:
            torque.Set(- (M_phys / thickness / I_edge) )
        else:
            torque.Set(M_phys / thickness / I_edge)

        # Solve nonlinear system with Newton               
        NewtonMinimization(a=bfa,
                           u=gf_u,
                           printing=False,
                           inverse="sparsecholesky",
                           maxerr=1e-10, maxit=30,
                           linesearch=False)                              

        # Update kinematic variables (velocity, acceleration)
        gf_v.vec[:] = 2/tau * (gf_u.vec-gf_uold.vec) - gf_vold.vec
        gf_a.vec[:] = 2/tau * (gf_v.vec-gf_vold.vec) - gf_aold.vec

        scene.Redraw()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,6))
plt.plot(t_vals, np.rad2deg(theta_vals))
plt.axhline(q0_deg, color='r', linestyle='--', label='Initial Angle')
#plt.axhline(-q0_deg, color='r', linestyle='--')
plt.xlabel("Time (s)")
plt.ylabel("Angle (degrees)")
plt.title("Pendulum Angle Over Time")
plt.grid()
plt.legend()
plt.show()